# GPU 程序计时与测量：方法总结

> 适用：测任何 GPU 代码的耗时与显存（算子、层、训练步、推理）。
> 一句话：**GPU 是异步的，所以必须同步、预热、重复、控制变量，并且先有预测再去测。**

---

## 1. 为什么不能直接 `time.time()`

| 问题 | 机理 | 后果 |
|---|---|---|
| 异步发射 | Python 调用只是把 kernel 放进队列就返回 | 不同步量到的是发射时间，偏小几十倍 |
| 首次迭代非稳态 | cuBLAS 初始化、kernel 懒加载、allocator 首次分配、`torch.compile` 编译、autotune | 首次比稳态慢几倍到几百倍 |
| 频率与温度 | GPU 随负载、温度调频 | 前后几次不一致 |
| 其他进程 | 卡不空闲 | 抖动大，数据不可信 |
| 缓存状态 | 微基准反复用同一份输入，L2 常驻 | 带宽测得偏乐观 |

---

## 2. 通用六步流程

1. **定义被测单元**：无参函数 `fn()`，只含要测的内容。数据生成、H2D、`.item()`、打印都放在外面。
2. **固定条件**：设备独占、dtype、形状、种子、TF32 开关、`train/eval`、`no_grad` 与否。
3. **预热**：不计时地跑若干次，让初始化、编译、allocator、频率进入稳态。
4. **同步后计时**：CUDA event，或计时前后各 `synchronize`。
5. **重复并统计**：中位数为主，同时记最小、最大、抖动。
6. **算派生指标，对照预测**：TFLOP/s、GB/s、tokens/s，与理论上限或预测比。

---

## 3. 关键 API

| API | 用途 | 注意 |
|---|---|---|
| `torch.cuda.synchronize()` | 等 GPU 上已发射的工作全部完成 | 预热结束、计时边界必须调用 |
| `torch.cuda.Event(enable_timing=True)` | GPU 侧时间戳 | `e.record()` 入当前 stream；读 `elapsed_time` 前保证 `e` 已完成 |
| `time.perf_counter()` | CPU 墙钟 | 仅在前后同步时有意义，用于连续吞吐 |
| `torch.no_grad()` / `inference_mode()` | 关 autograd | 测纯 forward 必须用 |
| `opt.zero_grad(set_to_none=True)` | 清梯度 | 与真实训练一致 |
| `torch.cuda.reset_peak_memory_stats()` | 重置峰值计数 | 每个阶段开始前调用 |
| `torch.cuda.max_memory_allocated()` | 张量实际占用峰值 | 对照理论估算 |
| `torch.cuda.max_memory_reserved()` | allocator 持有峰值 | 与上一项之差反映碎片 |
| `torch.utils.benchmark.Timer` | 官方计时器，自动预热与同步 | 快速测小函数 |
| `triton.testing.do_bench` | 微基准，默认清 L2 | 单 kernel 对比 |
| `torch.cuda.CUDAGraph` | 消除发射开销 | 区分"kernel 慢"与"发射慢" |

---

## 4. 两种计时方式，回答不同问题

| 方式 | 做法 | 回答 | 特点 |
|---|---|---|---|
| 逐次 event | 每次迭代前后各一个 event，每次之间同步 | 一步本身的 GPU 耗时（延迟） | 不含步间 CPU 空隙，稳定，适合与预测比 |
| 连续墙钟 | 连跑 N 步，仅头尾同步，总时间除以 N | 真实吞吐 | 含 CPU 发射与流水重叠，tokens/s 用它 |

两个数都记。`wall` 明显大于 `median` → 有 CPU 空隙（发射受限）；明显小于 → 步间有重叠。

---

## 5. 通用模板

```python
import torch, statistics, time

def gpu_time(fn, warmup=5, iters=20, flush_l2=False):
    """每次调用的 GPU 耗时(ms)。微基准用 flush_l2=True。"""
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    if flush_l2:
        buf = torch.empty(256 * 2**20, dtype=torch.uint8, device="cuda")  # 须大于 L2
    S = [torch.cuda.Event(enable_timing=True) for _ in range(iters)]
    E = [torch.cuda.Event(enable_timing=True) for _ in range(iters)]
    for i in range(iters):
        if flush_l2:
            buf.zero_()                      # 在 start 之前，不计入
        S[i].record(); fn(); E[i].record()
    torch.cuda.synchronize()                 # 全部记录完再同步，读取才安全
    return [s.elapsed_time(e) for s, e in zip(S, E)]

def wall_ms(fn, warmup=5, iters=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1e3

def summarize(ts):
    ts = sorted(ts); med = statistics.median(ts)
    return dict(median=med, min=ts[0], max=ts[-1], jitter=(ts[-1] - ts[0]) / med)
```

显存：

```python
torch.cuda.reset_peak_memory_stats()
fn(); torch.cuda.synchronize()
peak_gib = torch.cuda.max_memory_allocated() / 2**30
```

---

## 6. 派生指标

| 指标 | 公式 | 对照 |
|---|---|---|
| 达到算力 | FLOPs / t | 硬件峰值 P（fp32、TF32、bf16 各不同） |
| 达到带宽 | bytes moved / t | 硬件带宽峰值 |
| 吞吐 | B·T / t | tokens/s，用连续墙钟的 t |
| 利用率 | 达到值 ÷ 峰值 | 越接近 1 越没有优化空间 |
| 理想时间 | max(FLOPs/P, bytes/BW) | roofline 下限，**测量值不应低于它** |

---

## 7. 控制变量清单

- [ ] 设备独占：`nvidia-smi` 确认空闲，`CUDA_VISIBLE_DEVICES` 隔离
- [ ] 精度一致：TF32 显式设置；bf16 autocast 确认生效
- [ ] 形状固定：`torch.compile` 下形状变化会重编译
- [ ] 输入常驻 GPU：生成与 H2D 在计时外
- [ ] 计时区内无 `.item()`、`.cpu()`、`print`、`.tolist()`（会隐式同步）
- [ ] 模式一致：`train/eval`、`no_grad` 与要比的场景相同
- [ ] 固定种子并保存配置
- [ ] 预热覆盖所有首次开销（含 compile、优化器状态分配）

---

## 8. 统计与抖动

- 报告**中位数**，不是平均值（平均值被偶发慢迭代拉高）。
- 同时报最小值（近似无干扰的最好情况）与最大值（暴露干扰）。
- 抖动 = (max − min) / median，超过约 10% 先查卡是否空闲。
- 次数：单步几十 ms 以上，20 次够；微秒级 kernel 要内层循环摊薄发射开销或用 CUDA graph。
- 同一测量至少跑两遍。

---

## 9. 微基准 vs 宏基准

| | 微基准（单 kernel / 算子） | 宏基准（训练步 / 推理请求） |
|---|---|---|
| 目标 | 一个 kernel 离硬件极限多远 | 端到端真实速度 |
| L2 缓存 | 需要清 | 不用 |
| 发射开销 | 会主导，需摊薄或用 graph | 是真实成本，要保留 |
| 工具 | `do_bench`、`Timer`、ncu | benchmark 脚本、nsys |
| 输入 | 多份轮换，避免缓存命中 | 真实数据 |

---

## 10. 测量本身的正确性检查

先确认测量没问题，再解释结果。

| 检查 | 期望 | 不满足说明 |
|---|---|---|
| 时间不低于理想时间 | 测量值 ≥ roofline 下限 | 漏了同步，或结果没被使用被优化掉 |
| 批大小翻倍（GPU 受限） | 耗时约翻倍 | 几乎不变 → 发射受限或固定开销 |
| 去掉同步再测一遍 | 数字突然极小 | 证明原来的同步有效 |
| iters 20 → 100 | 中位数基本不变 | 未进稳态或有周期性干扰 |
| 两次独立运行 | 中位数差 < 3% | 环境不稳定 |

---

## 11. 常见坑

| 坑 | 后果 | 处理 |
|---|---|---|
| 没同步就读时间 | 量到发射时间 | event 加 `synchronize` |
| 预热不够 | 前几次拉高统计 | 预热到稳定；compile 后多跑几次 |
| 计时区里 `.item()` | 隐式同步，破坏流水 | 挪到计时区外 |
| 编译、懒加载计入 | 不可复现 | 预热包含它们 |
| 只报平均值 | 被离群值扭曲 | 中位数 + 最小最大 |
| 卡不空闲 | 抖动大 | 先 `nvidia-smi` |
| 优化器状态首次分配 | 第一步显存与耗时偏大 | 预热 ≥ 1 步，预热后再重置峰值 |
| 结果未被使用 | 被消除，时间趋近 0 | 使用或保存输出 |
| fwd 与 train 模式混测 | 不可比 | 统一 `no_grad` 与否 |
| profiler 下的时间当性能 | profiler 有开销 | 时间只信不加 profiler 的 benchmark |
| 训练模式 loss 变 NaN | 数值异常可能改变耗时 | 记录 loss 有限性，降低 lr |

---

## 12. 记录什么，测完怎么用

**写进 JSON 的内容**：设备名、`torch` 与 CUDA 版本、模型配置、形状、dtype、是否 compile、预热与迭代次数、原始每次耗时、峰值显存、git commit。

**测完的使用顺序**：

1. 对照预测：比值 0.8 到 1.25 视为预测成立。
2. 超出范围就从粗到细找原因：
   - benchmark → **多快**
   - nsys → **时间花在哪**、GPU 是否在等 CPU
   - ncu → **某个 kernel 为什么慢**（算力还是带宽受限）
3. GPU 利用率高才值得优化 kernel；利用率低先减少 kernel 数或消除发射开销（融合、`torch.compile`、CUDA graph）。

---

## 13. 项目脚本速查（cs336 `benchmark.py`）

- 三个模式：`fwd`（`no_grad` 前向）、`fwdbwd`（前向 + loss + 反向）、`full`（完整训练步含 optimizer）。
- 每个模式输出：median / min / max、jitter、wall、tok/s、达到 TFLOP/s、峰值 GiB，并存 JSON。
- FLOPs 公式（forward，不含逐元素）：
  `L·8·N·d² + L·6·N·d·d_ff + L·4·N·T·d + 2·N·d·V`，其中 `N = B·T`；`fwdbwd` 与 `full` 取 3 倍。
- small（L=12, d=768, d_ff=3072, V=10000, B=4, T=512）：forward 534.0 GFLOP，一步约 1.60 TFLOP。
- 运行：
  ```bash
  CUDA_VISIBLE_DEVICES=0 uv run python scripts/benchmark.py --model small --ctx 512 --batch 4
  ```

---

## 14. 最小核对清单

- [ ] 被测函数不含数据生成、拷贝、`.item()`、打印
- [ ] 预热足够，编译与分配已完成
- [ ] 用 event 或前后同步计时，没有裸 `time.time()`
- [ ] 报告中位数、最小、最大、抖动
- [ ] 区分逐次延迟与连续吞吐
- [ ] 每个阶段前重置显存峰值
- [ ] 通过正确性检查（不低于理想时间、批大小缩放）
- [ ] 配置与环境已保存
- [ ] 与预测对照，差异有归因